# 152 — Component Specialization Analysis

What does each model component specialize in? Some attention heads consistently
do one thing (e.g., attend to the previous token), while some MLP neurons fire
only for specific inputs. This module profiles, measures, and classifies
component specialization.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.component_specialization import (
    head_function_profile,
    mlp_specialization,
    component_consistency,
    specialization_spectrum,
    component_redundancy,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)

key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

tokens = jnp.arange(15)
tokens_list = [
    jnp.array([1, 10, 20, 30, 40, 50, 60]),
    jnp.array([5, 15, 25, 35, 45, 55, 65]),
    jnp.array([2, 12, 22, 32, 42, 52, 62]),
    jnp.array([8, 18, 28, 38, 48, 58, 68]),
]

## Head Function Profiling

Classify each attention head by its dominant behavior: self-attention,
previous-token, or first-token. Also reports uniformity (how spread out
the attention is).

In [ ]:
result = head_function_profile(model, tokens)

print("Function distribution:", result['function_distribution'])
print()
for h in result['per_head']:
    print(f"L{h['layer']}H{h['head']}: dominant={h['dominant_function']:16s} "
          f"self={h['self_attention']:.3f} prev={h['previous_token']:.3f} "
          f"first={h['first_token']:.3f} uniform={h['uniformity']:.3f}")

## MLP Neuron Specialization

Measure how selectively each MLP neuron activates. A specialized neuron fires
for specific inputs only; a generalist fires for everything.

In [ ]:
result = mlp_specialization(model, tokens_list, layer=0, top_k=5)

print(f"Layer {result['layer']}: mean_spec={result['mean_specialization']:.3f}")
print(f"Dead neurons: {result['n_dead']}, Always active: {result['n_always_active']}\n")

print("Most specialized:")
for n in result['most_specialized']:
    bar = '#' * int(n['specialization'] * 20)
    print(f"  neuron {n['neuron']:3d}: spec={n['specialization']:.3f} "
          f"freq={n['activation_frequency']:.3f} {bar}")

print("\nLeast specialized:")
for n in result['least_specialized']:
    bar = '#' * int(n['specialization'] * 20)
    print(f"  neuron {n['neuron']:3d}: spec={n['specialization']:.3f} "
          f"freq={n['activation_frequency']:.3f} {bar}")

## Component Consistency

Does a component produce the same output regardless of input? High consistency
means the component outputs a near-constant direction; low means it is
input-sensitive.

In [ ]:
result = component_consistency(model, tokens_list)

print(f"Most consistent: {result['most_consistent']}")
print(f"Least consistent: {result['least_consistent']}\n")

for c in result['per_component']:
    bar = '#' * int(c['consistency'] * 30)
    print(f"{c['component']:16s}: consistency={c['consistency']:.3f}  "
          f"deviation={c['mean_deviation']:.4f}  norm={c['mean_norm']:.4f}  {bar}")

## Specialization Spectrum

Place each component on a specialist-generalist spectrum. Specialists produce
very different outputs for different inputs (high diversity relative to norm);
generalists produce similar outputs.

In [ ]:
result = specialization_spectrum(model, tokens_list)

print(f"Specialists: {result['n_specialists']}, Generalists: {result['n_generalists']}\n")

for c in result['per_component']:
    tag = c['classification'].upper()
    bar = '#' * int(c['specialization_score'] * 30)
    print(f"{c['component']:10s}: [{tag:11s}] score={c['specialization_score']:.3f} "
          f"diversity={c['output_diversity']:.4f}  {bar}")

## Component Redundancy

Find pairs of components that produce nearly identical outputs (high cosine
similarity). Redundant components suggest the model could be pruned.

In [ ]:
result = component_redundancy(model, tokens)

print(f"Components: {result['n_components']}")
print(f"Redundant pairs (|cos| > 0.8): {result['n_redundant_pairs']}\n")

for pair in result['redundant_pairs']:
    print(f"  {pair['component_a']} <-> {pair['component_b']}: "
          f"cos_sim={pair['cosine_similarity']:.4f}")